# Module 10.1: Reinforcement Learning with GANs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_10_RL_For_Image_Generation/01_RL_With_GANs/rl_with_gans.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:

1. **Understand** the mathematical connection between GANs and reinforcement learning
2. **Formulate** GAN hyperparameter scheduling as a Markov Decision Process (MDP)
3. **Implement** a simple GAN on synthetic data with an RL meta-controller
4. **Train** a policy gradient agent that learns to adjust GAN training dynamics
5. **Compare** RL-guided GAN training against fixed-hyperparameter baselines

---

## Prerequisites

- Familiarity with GAN architecture (generator + discriminator)
- Understanding of policy gradient methods (REINFORCE)
- Basic PyTorch proficiency

In [ ]:
!pip install torch torchvision numpy matplotlib scikit-image

In [ ]:
# Download REAL open-source dataset for image generation (TINY - under 200MB)
import torchvision
import torchvision.transforms as transforms

# MNIST for GAN training (classic, tiny, real)
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5], [0.5])])
mnist = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
print(f"✅ MNIST: {len(mnist)} real images for GAN training (11MB)")

# FashionMNIST for more complex generation
fashion = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
print(f"✅ FashionMNIST: {len(fashion)} real clothing images (30MB)")

# CIFAR-10 for color image generation
transform_color = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5]*3, [0.5]*3)])
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_color)
print(f"✅ CIFAR-10: {len(cifar10)} real color photos for generation (170MB)")

print(f"\n📦 Total download: ~211MB")
print(f"   NO CelebA needed - MNIST/FashionMNIST/CIFAR-10 are perfect for learning!")
print(f"   MNIST → simple GAN, FashionMNIST → medium, CIFAR-10 → color generation")

## Deep Derivation: GAN Theory and RL Connection

### Step 1: GAN Minimax Objective
$$\min_G \max_D V(D, G) = E_{x\sim p_{data}}[\log D(x)] + E_{z\sim p_z}[\log(1 - D(G(z)))]$$

### Step 2: Optimal Discriminator (Fixed $G$)
For fixed $G$, the optimal $D$ maximizes:
$$V(D) = \int_x \left[p_{data}(x)\log D(x) + p_g(x)\log(1-D(x))\right] dx$$

Take derivative and set to zero:
$$\frac{p_{data}(x)}{D(x)} - \frac{p_g(x)}{1-D(x)} = 0$$

Solving: $p_{data}(1-D^*) = p_g \cdot D^*$

$$D^*(x) = \frac{p_{data}(x)}{p_{data}(x) + p_g(x)} \quad \blacksquare$$

### Step 3: With Optimal D*, Generator Minimizes JS Divergence
Substituting $D^*$ back:
$$V(G, D^*) = E_{x\sim p_{data}}\left[\log\frac{p_{data}}{p_{data}+p_g}\right] + E_{x\sim p_g}\left[\log\frac{p_g}{p_{data}+p_g}\right]$$

$$= -\log 4 + 2 \cdot D_{JS}(p_{data} \| p_g)$$

where Jensen-Shannon divergence:
$$D_{JS}(p\|q) = \frac{1}{2}D_{KL}\left(p \middle\| \frac{p+q}{2}\right) + \frac{1}{2}D_{KL}\left(q \middle\| \frac{p+q}{2}\right)$$

**Global minimum:** $V^* = -\log 4$ when $p_g = p_{data}$ (i.e., $D_{JS} = 0$). $\blacksquare$

### Step 4: Training Instability Analysis
**Mode collapse:** $G$ maps many $z$ to same $x$ that fools $D$.

**Vanishing gradients:** When $D$ is too good, $D(G(z)) \approx 0$:
$$\nabla_G \log(1-D(G(z))) \approx \nabla_G \log 1 = 0$$

**Fix (non-saturating loss):** Generator maximizes $\log D(G(z))$ instead.

### Step 5: RL Perspective on GANs
**Generator** = policy $\pi_\theta$ that generates pixel sequences.
**Discriminator** = reward model $R_\phi$.
**Training** = policy gradient with $R(x) = \log D(x)$:
$$\nabla_\theta J = E_{x\sim G_\theta}[\nabla_\theta \log \pi_\theta(x) \cdot \log D_\phi(x)]$$

This is SeqGAN! The RL-GAN connection is fundamental.

### Step 6: FID Score (Frechet Inception Distance)
$$\text{FID} = \|\mu_r - \mu_g\|^2 + \text{Tr}\left(\Sigma_r + \Sigma_g - 2(\Sigma_r \Sigma_g)^{1/2}\right)$$

This is the Wasserstein-2 distance between Gaussians $\mathcal{N}(\mu_r, \Sigma_r)$ and $\mathcal{N}(\mu_g, \Sigma_g)$.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Mathematical Foundation

### 1.1 The GAN Objective

A Generative Adversarial Network consists of two competing networks:

- **Generator** $G_\theta: \mathcal{Z} \to \mathcal{X}$ maps latent noise $z \sim p_z(z)$ to data space
- **Discriminator** $D_\phi: \mathcal{X} \to [0,1]$ distinguishes real from generated data

The minimax objective is:

$$\min_\theta \max_\phi \; V(G_\theta, D_\phi) = \mathbb{E}_{x \sim p_{\text{data}}}[\log D_\phi(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D_\phi(G_\theta(z)))]$$

At the Nash equilibrium, $D^*(x) = \frac{p_{\text{data}}(x)}{p_{\text{data}}(x) + p_g(x)}$ and $p_g = p_{\text{data}}$.

### 1.2 The Training Instability Problem

GAN training is notoriously unstable. Key failure modes include:

| Failure Mode | Mathematical Signature |
|---|---|
| Mode collapse | $H(p_g) \ll H(p_{\text{data}})$ |
| Vanishing gradients | $\|\nabla_\theta \mathcal{L}_G\| \to 0$ when $D$ is too strong |
| Oscillation | $\mathcal{L}_D, \mathcal{L}_G$ cycle without convergence |

### 1.3 RL Formulation for GAN Meta-Learning

We cast hyperparameter scheduling as an MDP $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$:

**State** $s_t$ captures training statistics at epoch $t$:
$$s_t = \left[\mathcal{L}_D^{(t)}, \; \mathcal{L}_G^{(t)}, \; \left\|\nabla_\theta \mathcal{L}_G\right\|, \; \left\|\nabla_\phi \mathcal{L}_D\right\|, \; \hat{Q}_t \right]$$

where $\hat{Q}_t$ is a quality proxy (e.g., discriminator confusion $D(G(z)) \approx 0.5$).

**Actions** $a_t$ adjust hyperparameters:
$$a_t \in \{\text{lr}_{\text{up}}, \text{lr}_{\text{down}}, \text{D-steps}_{\text{up}}, \text{D-steps}_{\text{down}}, \text{noop}\}$$

**Reward** reflects generation quality improvement:
$$r_t = -\Delta\text{FID}_t = -(\text{FID}_{t} - \text{FID}_{t-1})$$

or a proxy:
$$r_t = -|D(G(z)) - 0.5| + \lambda \cdot \text{diversity}(G(z_1), \ldots, G(z_k))$$

**Policy** $\pi_\psi(a_t | s_t)$ is trained with REINFORCE:
$$\nabla_\psi J(\psi) = \mathbb{E}_{\tau \sim \pi_\psi}\left[\sum_{t=0}^{T} \nabla_\psi \log \pi_\psi(a_t|s_t) \left(\sum_{t'=t}^{T} \gamma^{t'-t} r_{t'} - b(s_t)\right)\right]$$

where $b(s_t)$ is a baseline (e.g., running average of returns).

## 2. Synthetic Data Distribution

We use a mixture of 2D Gaussians as our target data distribution. This allows us to clearly visualize mode coverage and collapse.

In [ ]:
def generate_ring_data(n_samples=1000, n_modes=8, radius=2.0, std=0.1):
    """Generate 2D data from a ring of Gaussian modes."""
    angles = np.linspace(0, 2 * np.pi, n_modes, endpoint=False)
    centers = np.stack([radius * np.cos(angles), radius * np.sin(angles)], axis=1)
    
    samples = []
    for _ in range(n_samples):
        idx = np.random.randint(n_modes)
        sample = centers[idx] + std * np.random.randn(2)
        samples.append(sample)
    return np.array(samples, dtype=np.float32), centers

real_data, mode_centers = generate_ring_data(n_samples=2000)

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.scatter(real_data[:, 0], real_data[:, 1], alpha=0.3, s=5, label='Real data')
ax.scatter(mode_centers[:, 0], mode_centers[:, 1], c='red', s=100, marker='x', label='Mode centers')
ax.set_title('Target Distribution: Ring of 8 Gaussians')
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5)
ax.set_aspect('equal')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Deep Derivation: Wasserstein Distance and GAN Training Stability

### Step 1: Wasserstein-1 Distance (Earth Mover's Distance)

The Wasserstein-1 distance between distributions $p$ and $q$:

$$W_1(p, q) = \inf_{\gamma \in \Pi(p,q)} \mathbb{E}_{(x,y) \sim \gamma}[\|x - y\|]$$

where $\Pi(p,q)$ is the set of all joint distributions with marginals $p$ and $q$.

### Step 2: Kantorovich-Rubinstein Duality

**Theorem:** $W_1(p, q) = \sup_{\|f\|_L \leq 1} \left[\mathbb{E}_{x \sim p}[f(x)] - \mathbb{E}_{x \sim q}[f(x)]\right]$

where the supremum is over all 1-Lipschitz functions ($|f(x) - f(y)| \leq \|x - y\|$).

**Proof sketch:**

The primal (transport) and dual (Lipschitz) formulations are connected by LP duality. The optimal $f^*$ is the **Kantorovich potential**, which represents the "price" of transporting mass from $q$ to $p$.

For discrete distributions: this reduces to the LP dual of the optimal transport problem. Strong duality holds by Slater's condition (feasible point exists). $\blacksquare$

### Step 3: WGAN Objective

Replace the GAN discriminator with a **critic** $f_w$ (no sigmoid):

$$\min_G \max_{f_w \in \mathcal{F}_L} \mathbb{E}_{x \sim p_{\text{data}}}[f_w(x)] - \mathbb{E}_{z \sim p_z}[f_w(G(z))]$$

where $\mathcal{F}_L$ = set of 1-Lipschitz functions.

**Why Wasserstein is better than JS divergence:**

When $p_g$ and $p_{\text{data}}$ have disjoint supports (common in early training):

$$D_{JS}(p_g \| p_{\text{data}}) = \log 2 \quad \text{(constant — no gradient!)}$$

$$W_1(p_g, p_{\text{data}}) = d(p_g, p_{\text{data}}) \quad \text{(proportional to distance — gradient exists!)}$$

**Proof $D_{JS}$ is constant for disjoint supports:**

If $\text{supp}(p) \cap \text{supp}(q) = \emptyset$, then the mixture $m = \frac{p+q}{2}$ satisfies:

$$D_{KL}(p \| m) = \int p \log \frac{p}{(p+q)/2} = \int p \log \frac{2p}{p} = \log 2$$

Similarly $D_{KL}(q \| m) = \log 2$. So $D_{JS} = \frac{1}{2}(\log 2 + \log 2) = \log 2$. $\blacksquare$

### Step 4: Lipschitz Constraint Enforcement

**Weight clipping (WGAN):** Clip $w \in [-c, c]$ after each update.

**Problem:** Constrains critic capacity. If $c$ is too small, the critic underfits.

**Gradient penalty (WGAN-GP):**

$$\mathcal{L}_{\text{GP}} = \lambda \mathbb{E}_{\hat{x} \sim p_{\hat{x}}}\left[(|\nabla_{\hat{x}} f_w(\hat{x})\|_2 - 1)^2\right]$$

where $\hat{x} = \epsilon x + (1-\epsilon) G(z)$ for $\epsilon \sim U[0,1]$.

**Proof GP enforces Lipschitz constraint:**

A differentiable function is 1-Lipschitz iff $\|\nabla f(x)\| \leq 1$ everywhere. The penalty term pushes $\|\nabla f\|$ toward 1 along interpolation lines between real and fake samples. $\blacksquare$

### Step 5: Mode Collapse — Formal Analysis

**Definition:** Mode collapse occurs when $\text{supp}(p_g) \subsetneq \text{supp}(p_{\text{data}})$.

**Detection criterion:** For Gaussian mixture target with $K$ modes:

$$\text{Coverage} = \frac{|\{k : \min_{i} \|G(z_i) - \mu_k\| < \tau\}|}{K}$$

**Why mode collapse happens:**

For the minimax game, the generator's optimal response to a perfect discriminator is to place all mass on the point $x^* = \arg\max_{x} p_{\text{data}}(x)$ — the mode of the data. The alternating optimization gets stuck in this local equilibrium.

The RL controller combats mode collapse by adjusting discriminator training steps: fewer D-steps → weaker D → generator has room to explore multiple modes.

## 3. GAN Architecture

We build a simple MLP-based GAN for the 2D distribution.

In [ ]:
class Generator(nn.Module):
    def __init__(self, noise_dim=16, hidden_dim=128, output_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.net(x)


def compute_quality_proxy(gen, disc, noise_dim, n_samples=500):
    """Proxy for generation quality: how close D(G(z)) is to 0.5 + mode coverage."""
    gen.eval()
    disc.eval()
    with torch.no_grad():
        z = torch.randn(n_samples, noise_dim, device=device)
        fake = gen(z)
        d_scores = disc(fake)
        confusion = 1.0 - torch.abs(d_scores - 0.5).mean().item() * 2
        
        fake_np = fake.cpu().numpy()
        covered_modes = 0
        for center in mode_centers:
            dists = np.linalg.norm(fake_np - center, axis=1)
            if np.sum(dists < 0.5) > 2:
                covered_modes += 1
        coverage = covered_modes / len(mode_centers)
    gen.train()
    disc.train()
    return confusion, coverage


def compute_gradient_norm(model):
    total_norm = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total_norm += p.grad.data.norm(2).item() ** 2
    return total_norm ** 0.5

print('GAN components defined.')

## 4. GAN Training Function

A configurable GAN training step that accepts hyperparameters from the RL controller.

In [ ]:
def train_gan_step(gen, disc, opt_g, opt_d, real_batch, noise_dim, d_steps=1):
    """One training step: d_steps discriminator updates + 1 generator update."""
    batch_size = real_batch.size(0)
    criterion = nn.BCELoss()
    
    d_loss_total = 0.0
    for _ in range(d_steps):
        opt_d.zero_grad()
        real_labels = torch.ones(batch_size, 1, device=device)
        fake_labels = torch.zeros(batch_size, 1, device=device)
        
        d_real = disc(real_batch)
        loss_real = criterion(d_real, real_labels)
        
        z = torch.randn(batch_size, noise_dim, device=device)
        fake = gen(z).detach()
        d_fake = disc(fake)
        loss_fake = criterion(d_fake, fake_labels)
        
        d_loss = loss_real + loss_fake
        d_loss.backward()
        opt_d.step()
        d_loss_total += d_loss.item()
    
    d_grad_norm = compute_gradient_norm(disc)
    
    opt_g.zero_grad()
    z = torch.randn(batch_size, noise_dim, device=device)
    fake = gen(z)
    d_fake = disc(fake)
    g_loss = criterion(d_fake, torch.ones(batch_size, 1, device=device))
    g_loss.backward()
    opt_g.step()
    
    g_grad_norm = compute_gradient_norm(gen)
    
    return {
        'd_loss': d_loss_total / d_steps,
        'g_loss': g_loss.item(),
        'd_grad_norm': d_grad_norm,
        'g_grad_norm': g_grad_norm
    }

print('GAN training step defined.')

## 5. RL Controller (Policy Gradient)

The RL agent observes GAN training statistics and decides how to adjust hyperparameters.

### Action Space

| Action Index | Description |
|---|---|
| 0 | No change (noop) |
| 1 | Increase learning rate ($\times 1.2$) |
| 2 | Decrease learning rate ($\times 0.8$) |
| 3 | Increase D-steps (+1, max 5) |
| 4 | Decrease D-steps (-1, min 1) |

## Deep Derivation: GAN Training Dynamics and Nash Equilibrium

### Step 1: Two-Player Game Formulation

GAN training is a zero-sum game. Define payoff functions:

$$u_D(\phi, \theta) = \mathbb{E}_{x \sim p_d}[\log D_\phi(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D_\phi(G_\theta(z)))]$$

$$u_G(\phi, \theta) = -u_D(\phi, \theta)$$

A **Nash equilibrium** $(\phi^*, \theta^*)$ satisfies:

$$u_D(\phi^*, \theta^*) \geq u_D(\phi, \theta^*) \quad \forall \phi$$
$$u_G(\phi^*, \theta^*) \geq u_G(\phi^*, \theta) \quad \forall \theta$$

### Step 2: Gradient Dynamics Analysis

Simultaneous gradient descent updates:

$$\phi_{t+1} = \phi_t + \alpha \nabla_\phi u_D$$
$$\theta_{t+1} = \theta_t - \alpha \nabla_\theta u_D$$

The Jacobian of this dynamical system:

$$\mathbf{J} = \begin{bmatrix} \nabla_\phi^2 u_D & \nabla_\phi \nabla_\theta u_D \\ -\nabla_\theta \nabla_\phi u_D & -\nabla_\theta^2 u_D \end{bmatrix}$$

**Stability analysis:** The eigenvalues of $\mathbf{J}$ determine convergence behavior:
- Real negative eigenvalues → convergence
- Imaginary eigenvalues → oscillation (common in GANs!)
- Real positive eigenvalues → divergence

**Proof of oscillation for bilinear games:**

For $u_D = \phi \theta$ (simplest bilinear game): $\mathbf{J} = \begin{bmatrix} 0 & 1 \\ -1 & 0 \end{bmatrix}$

Eigenvalues: $\lambda = \pm i$ (purely imaginary) → perpetual oscillation around the Nash equilibrium. $\blacksquare$

### Step 3: Spectral Normalization — Controlling Discriminator

**Spectral norm:** $\sigma(W) = \max_{\|h\|=1} \|Wh\|_2$ (largest singular value).

**Spectral normalization:** Replace $W \leftarrow W / \sigma(W)$ after each update.

**Proof this makes each layer 1-Lipschitz:**

For a linear layer $f(x) = Wx$: $\|f(x) - f(y)\| = \|W(x-y)\| \leq \sigma(W)\|x-y\|$.

After normalization: $\sigma(W/\sigma(W)) = 1$, so $f$ is 1-Lipschitz.

For the full network $D = f_L \circ \cdots \circ f_1$: $\|D(x) - D(y)\| \leq \prod_l \sigma(W_l/\sigma(W_l)) \cdot \|x-y\| = \|x-y\|$.

The discriminator is globally 1-Lipschitz — satisfying the WGAN constraint automatically! $\blacksquare$

### Step 4: RL Controller Reward as GAN Quality Proxy

Our RL reward uses:

$$r_t = (Q_t - Q_{t-1}) \cdot 10, \quad Q_t = 0.5 \cdot \text{confusion}_t + 0.5 \cdot \text{coverage}_t$$

**Confusion metric:** $\text{confusion} = 1 - 2|D(G(z)) - 0.5|$. This is maximized when $D(G(z)) = 0.5$ (discriminator cannot distinguish real from fake).

**Proof confusion $= 1$ at equilibrium:**

At Nash equilibrium, $D^*(x) = 0.5$ for all $x$ in $\text{supp}(p_d) \cup \text{supp}(p_g)$. Therefore $|D^*(G(z)) - 0.5| = 0$ and confusion $= 1$. $\blacksquare$

**Coverage metric:** Measures mode diversity. Together, confusion + coverage prevents mode collapse (high confusion alone is achievable by collapsing to one mode that perfectly fools $D$).

In [ ]:
class RLController(nn.Module):
    """Policy gradient agent for GAN hyperparameter control."""
    
    def __init__(self, state_dim=5, n_actions=5, hidden_dim=64):
        super().__init__()
        self.policy = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_actions)
        )
        self.saved_log_probs = []
        self.rewards = []
    
    def forward(self, state):
        return torch.softmax(self.policy(state), dim=-1)
    
    def select_action(self, state):
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        probs = self.forward(state_t)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        self.saved_log_probs.append(dist.log_prob(action))
        return action.item()


def apply_action(action, current_lr, current_d_steps):
    """Apply RL action to GAN hyperparameters."""
    new_lr = current_lr
    new_d_steps = current_d_steps
    
    if action == 1:
        new_lr = min(current_lr * 1.2, 1e-2)
    elif action == 2:
        new_lr = max(current_lr * 0.8, 1e-5)
    elif action == 3:
        new_d_steps = min(current_d_steps + 1, 5)
    elif action == 4:
        new_d_steps = max(current_d_steps - 1, 1)
    
    return new_lr, new_d_steps


def update_controller(controller, optimizer, gamma=0.99):
    """REINFORCE update with baseline subtraction."""
    R = 0
    returns = deque()
    for r in reversed(controller.rewards):
        R = r + gamma * R
        returns.appendleft(R)
    
    returns = torch.tensor(list(returns), dtype=torch.float32, device=device)
    if returns.std() > 1e-8:
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    
    policy_loss = []
    for log_prob, R in zip(controller.saved_log_probs, returns):
        policy_loss.append(-log_prob * R)
    
    optimizer.zero_grad()
    loss = torch.stack(policy_loss).sum()
    loss.backward()
    optimizer.step()
    
    controller.saved_log_probs.clear()
    controller.rewards.clear()
    return loss.item()

print('RL Controller defined.')

## 6. Training Loop: RL-Guided GAN

We train the GAN over multiple episodes. In each episode, the RL agent observes the GAN state, takes an action (adjusting hyperparameters), and receives a reward based on quality improvement.

In [ ]:
def run_rl_guided_gan(n_episodes=15, steps_per_episode=80, noise_dim=16, batch_size=256):
    """Run RL-guided GAN training."""
    controller = RLController().to(device)
    ctrl_optimizer = optim.Adam(controller.parameters(), lr=1e-3)
    
    all_quality_scores = []
    all_coverage_scores = []
    all_actions = []
    all_lrs = []
    all_d_steps_log = []
    
    real_tensor = torch.FloatTensor(real_data).to(device)
    
    best_gen_state = None
    best_quality = -float('inf')
    
    for episode in range(n_episodes):
        gen = Generator(noise_dim=noise_dim).to(device)
        disc = Discriminator().to(device)
        
        current_lr = 2e-4
        current_d_steps = 1
        
        opt_g = optim.Adam(gen.parameters(), lr=current_lr, betas=(0.5, 0.999))
        opt_d = optim.Adam(disc.parameters(), lr=current_lr, betas=(0.5, 0.999))
        
        episode_qualities = []
        episode_coverage = []
        prev_quality = 0.0
        
        for step in range(steps_per_episode):
            idx = np.random.randint(0, len(real_data), batch_size)
            real_batch = real_tensor[idx]
            
            stats = train_gan_step(gen, disc, opt_g, opt_d, real_batch, noise_dim, current_d_steps)
            
            confusion, coverage = compute_quality_proxy(gen, disc, noise_dim)
            quality = 0.5 * confusion + 0.5 * coverage
            
            state = [
                stats['d_loss'],
                stats['g_loss'],
                stats['g_grad_norm'],
                stats['d_grad_norm'],
                quality
            ]
            
            action = controller.select_action(state)
            
            reward = (quality - prev_quality) * 10.0
            controller.rewards.append(reward)
            
            current_lr, current_d_steps = apply_action(action, current_lr, current_d_steps)
            for pg in opt_g.param_groups:
                pg['lr'] = current_lr
            for pg in opt_d.param_groups:
                pg['lr'] = current_lr
            
            prev_quality = quality
            episode_qualities.append(quality)
            episode_coverage.append(coverage)
            all_actions.append(action)
            all_lrs.append(current_lr)
            all_d_steps_log.append(current_d_steps)
        
        ctrl_loss = update_controller(controller, ctrl_optimizer)
        
        final_q = episode_qualities[-1]
        if final_q > best_quality:
            best_quality = final_q
            best_gen_state = gen.state_dict()
        
        all_quality_scores.extend(episode_qualities)
        all_coverage_scores.extend(episode_coverage)
        
        if (episode + 1) % 3 == 0:
            print(f'Episode {episode+1}/{n_episodes} | '
                  f'Quality: {final_q:.3f} | Coverage: {episode_coverage[-1]:.3f} | '
                  f'LR: {current_lr:.6f} | D-steps: {current_d_steps} | '
                  f'Ctrl Loss: {ctrl_loss:.4f}')
    
    best_gen = Generator(noise_dim=noise_dim).to(device)
    best_gen.load_state_dict(best_gen_state)
    
    return {
        'quality': all_quality_scores,
        'coverage': all_coverage_scores,
        'actions': all_actions,
        'lrs': all_lrs,
        'd_steps': all_d_steps_log,
        'best_gen': best_gen
    }

print('Training RL-guided GAN...')
rl_results = run_rl_guided_gan()
print('Done!')

## 7. Baseline: Fixed-Hyperparameter GAN

In [ ]:
def run_fixed_gan(n_steps=1200, noise_dim=16, batch_size=256, lr=2e-4, d_steps=1):
    """Train a GAN with fixed hyperparameters as a baseline."""
    gen = Generator(noise_dim=noise_dim).to(device)
    disc = Discriminator().to(device)
    opt_g = optim.Adam(gen.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_d = optim.Adam(disc.parameters(), lr=lr, betas=(0.5, 0.999))
    
    real_tensor = torch.FloatTensor(real_data).to(device)
    
    qualities = []
    coverages = []
    
    for step in range(n_steps):
        idx = np.random.randint(0, len(real_data), batch_size)
        real_batch = real_tensor[idx]
        train_gan_step(gen, disc, opt_g, opt_d, real_batch, noise_dim, d_steps)
        
        confusion, coverage = compute_quality_proxy(gen, disc, noise_dim)
        quality = 0.5 * confusion + 0.5 * coverage
        qualities.append(quality)
        coverages.append(coverage)
    
    return {'quality': qualities, 'coverage': coverages, 'gen': gen}

print('Training fixed-hyperparameter baselines...')
fixed_results_1 = run_fixed_gan(lr=2e-4, d_steps=1)
print('  Baseline 1 (lr=2e-4, d_steps=1) done.')
fixed_results_2 = run_fixed_gan(lr=1e-3, d_steps=3)
print('  Baseline 2 (lr=1e-3, d_steps=3) done.')
print('All baselines done!')

## 8. Visualization & Comparison

## Deep Derivation: GAN Diversity Metrics and Information-Theoretic Analysis

### Step 1: Inception Score (IS)

$$\text{IS} = \exp\left(\mathbb{E}_{x \sim p_g}\left[D_{KL}(p(y|x) \| p(y))\right]\right)$$

where $p(y|x)$ is the Inception network's class prediction and $p(y) = \mathbb{E}_x[p(y|x)]$ is the marginal.

**High IS requires:**
1. Each sample has **confident** classification ($p(y|x)$ is peaked → high quality)
2. The marginal $p(y)$ is **uniform** ($\approx 1/K$ → high diversity)

**Proof IS decomposes into quality and diversity:**

$$\log \text{IS} = H(y) - \mathbb{E}_x[H(y|x)]$$

$H(y)$ measures diversity (uniform → $\log K$). $H(y|x)$ measures quality (low entropy → clear images). $\blacksquare$

### Step 2: FID — Fréchet Inception Distance (Detailed)

$$\text{FID} = \|\mu_r - \mu_g\|_2^2 + \text{Tr}\left(\Sigma_r + \Sigma_g - 2(\Sigma_r \Sigma_g)^{1/2}\right)$$

**Proof FID is the squared Wasserstein-2 distance between Gaussians:**

For $P = \mathcal{N}(\mu_1, \Sigma_1)$ and $Q = \mathcal{N}(\mu_2, \Sigma_2)$:

$$W_2^2(P, Q) = \|\mu_1 - \mu_2\|^2 + \mathcal{B}^2(\Sigma_1, \Sigma_2)$$

where $\mathcal{B}^2$ is the **Bures metric**: $\mathcal{B}^2(\Sigma_1, \Sigma_2) = \text{Tr}(\Sigma_1 + \Sigma_2 - 2(\Sigma_1^{1/2}\Sigma_2\Sigma_1^{1/2})^{1/2})$.

For commuting matrices ($\Sigma_1 \Sigma_2 = \Sigma_2 \Sigma_1$): $(\Sigma_1^{1/2}\Sigma_2\Sigma_1^{1/2})^{1/2} = (\Sigma_1\Sigma_2)^{1/2}$, giving the standard FID formula. $\blacksquare$

### Step 3: Precision and Recall for Generative Models

**Precision:** Fraction of generated samples that fall in the support of real data.

$$\text{Precision} = \mathbb{E}_{x \sim p_g}[\mathbb{1}[x \in \text{supp}(p_r)]]$$

**Recall:** Fraction of real data modes covered by the generator.

$$\text{Recall} = \mathbb{E}_{x \sim p_r}[\mathbb{1}[x \in \text{supp}(p_g)]]$$

**Mode collapse detection:** Precision stays high (generated samples are realistic) but Recall drops (missing modes).

### Step 4: Information-Theoretic GAN Analysis

**InfoGAN** maximizes mutual information $I(c; G(z, c))$ between latent code $c$ and generated output:

$$I(c; G(z, c)) = H(c) - H(c | G(z, c))$$

**Variational lower bound (since $H(c|G)$ is intractable):**

$$I(c; G(z, c)) \geq \mathbb{E}_{c, z}[\log Q(c | G(z, c))] + H(c) = L_I(G, Q)$$

where $Q(c|x)$ is an auxiliary network approximating the posterior.

**Connection to RL:** The mutual information objective is analogous to intrinsic motivation in RL — the generator is rewarded for producing outputs that reveal information about its input code, encouraging diverse generation. $\blacksquare$

In [ ]:
def smooth(values, window=30):
    """Simple moving average smoothing."""
    smoothed = []
    for i in range(len(values)):
        start = max(0, i - window)
        smoothed.append(np.mean(values[start:i+1]))
    return smoothed

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Quality comparison
ax = axes[0, 0]
ax.plot(smooth(rl_results['quality']), label='RL-Guided', linewidth=2)
ax.plot(smooth(fixed_results_1['quality']), label='Fixed (lr=2e-4, D=1)', linewidth=2, linestyle='--')
ax.plot(smooth(fixed_results_2['quality']), label='Fixed (lr=1e-3, D=3)', linewidth=2, linestyle=':')
ax.set_xlabel('Training Step')
ax.set_ylabel('Quality Score')
ax.set_title('Generation Quality Over Training')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Mode coverage
ax = axes[0, 1]
ax.plot(smooth(rl_results['coverage']), label='RL-Guided', linewidth=2)
ax.plot(smooth(fixed_results_1['coverage']), label='Fixed (lr=2e-4, D=1)', linewidth=2, linestyle='--')
ax.plot(smooth(fixed_results_2['coverage']), label='Fixed (lr=1e-3, D=3)', linewidth=2, linestyle=':')
ax.set_xlabel('Training Step')
ax.set_ylabel('Mode Coverage')
ax.set_title('Mode Coverage Over Training')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# RL actions histogram
ax = axes[1, 0]
action_names = ['Noop', 'LR Up', 'LR Down', 'D-steps Up', 'D-steps Down']
action_counts = [rl_results['actions'].count(i) for i in range(5)]
colors = ['#95a5a6', '#2ecc71', '#e74c3c', '#3498db', '#e67e22']
ax.bar(action_names, action_counts, color=colors)
ax.set_title('RL Action Distribution')
ax.set_ylabel('Count')
for i, v in enumerate(action_counts):
    ax.text(i, v + 5, str(v), ha='center', fontweight='bold')

# Learning rate trajectory
ax = axes[1, 1]
ax.plot(rl_results['lrs'], alpha=0.7, linewidth=1)
ax.set_xlabel('Training Step')
ax.set_ylabel('Learning Rate')
ax.set_title('RL-Scheduled Learning Rate')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

plt.suptitle('RL-Guided vs Fixed-Hyperparameter GAN Training', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
noise_dim = 16

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

generators = [
    (rl_results['best_gen'], 'RL-Guided GAN'),
    (fixed_results_1['gen'], 'Fixed (lr=2e-4, D=1)'),
    (fixed_results_2['gen'], 'Fixed (lr=1e-3, D=3)')
]

for ax, (gen, title) in zip(axes, generators):
    gen.eval()
    with torch.no_grad():
        z = torch.randn(1000, noise_dim, device=device)
        samples = gen(z).cpu().numpy()
    
    ax.scatter(real_data[:, 0], real_data[:, 1], alpha=0.15, s=3, c='blue', label='Real')
    ax.scatter(samples[:, 0], samples[:, 1], alpha=0.3, s=3, c='red', label='Generated')
    ax.scatter(mode_centers[:, 0], mode_centers[:, 1], c='black', s=80, marker='x', zorder=5)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-3.5, 3.5)
    ax.set_aspect('equal')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Generated Samples Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Analysis: Why RL Helps GAN Training

### Key Insights

The RL controller learns **adaptive training strategies** that respond to the GAN's current state:

1. **Early training**: The controller may increase D-steps to give the discriminator a head start, providing meaningful gradients to the generator

2. **Mid training**: As the generator improves, the controller balances D/G training to avoid the discriminator becoming too strong (vanishing gradients for $G$)

3. **Late training**: Fine-tuning the learning rate to enable convergence without oscillation

### Theoretical Justification

The optimal discriminator at any point satisfies:

$$D^*_\phi(x) = \frac{p_{\text{data}}(x)}{p_{\text{data}}(x) + p_g(x)}$$

The generator gradient depends on how well-calibrated the discriminator is. If $D$ is too far from $D^*$, the generator receives **biased gradients**. The RL controller implicitly learns to keep $D$ near $D^*$ by adjusting the training ratio.

The expected return the RL agent maximizes is:

$$J(\psi) = \mathbb{E}_{\pi_\psi}\left[\sum_{t=0}^{T} \gamma^t r_t\right] \approx \mathbb{E}_{\pi_\psi}\left[-\text{FID}(p_g^{(T)}, p_{\text{data}})\right]$$

This effectively optimizes the end-of-training quality through a sequence of hyperparameter decisions.

## Summary

In this notebook, we explored the intersection of Reinforcement Learning and Generative Adversarial Networks:

1. **GAN training as an MDP**: We formulated hyperparameter scheduling as a sequential decision problem where the state captures training dynamics and the reward reflects generation quality

2. **REINFORCE controller**: A lightweight policy gradient agent learns to adjust the learning rate and discriminator training steps based on observed GAN behavior

3. **Empirical comparison**: The RL-guided approach adapts to training dynamics, potentially achieving better mode coverage and stability than fixed hyperparameter schedules

4. **Practical implications**: This framework extends to more complex GANs (DCGAN, StyleGAN) and richer action spaces (architecture search, loss function selection)

### Next Steps

- Extend to image-based GANs (MNIST, CIFAR-10)
- Add architecture search actions (layer width, activation functions)
- Use PPO instead of REINFORCE for more stable controller training
- Integrate FID computation as a proper reward signal